# Hybrid E-Commerce Recommendation System
### Step 3 — Data Preprocessing & Feature Engineering

**Scope of this notebook:** cleaning, feature engineering, sparse matrix construction,
train/test splitting, and saving reusable artifacts. **No model training happens here.**

This notebook is self-contained: the first section re-links to the data (same cells as the
EDA notebook) so you can run this fresh in a new Colab session without re-doing Steps 1-2 manually.


## Linking Section: Re-acquire Data (same as EDA notebook, Step 1)

In [1]:
!pip install -q kaggle

### Upload Kaggle API token

In [2]:
import os
# Search competitions by keyword (equivalent of "datasets list -s")
!kaggle competitions list -s "h-and-m"
# List all files inside the competition (confirms exact filenames + sizes)
!kaggle competitions files -c h-and-m-personalized-fashion-recommendations
os.environ["KAGGLE_API_TOKEN"] = "KGAT_283606f7db27511d5bedbdc1c43c0695"

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [3]:
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f articles.csv
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f customers.csv
!kaggle competitions download -c h-and-m-personalized-fashion-recommendations -f transactions_train.csv

!ls -lh *.zip *.csv 2>/dev/null

articles.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
customers.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
transactions_train.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
-rw-r--r-- 1 root root  35M Aug  8 16:19 articles.csv
-rw-r--r-- 1 root root 4.3M Jan 17  2022 articles.csv.zip
-rw-r--r-- 1 root root 198M Aug  8 16:19 customers.csv
-rw-r--r-- 1 root root  98M Jan 17  2022 customers.csv.zip
-rw-r--r-- 1 root root 3.3G Aug  8 16:20 transactions_train.csv
-rw-r--r-- 1 root root 585M Jan 17  2022 transactions_train.csv.zip


### Extract

In [4]:
import zipfile
import os

for f in ["articles.csv.zip", "customers.csv.zip", "transactions_train.csv.zip"]:
    if os.path.exists(f):
        with zipfile.ZipFile(f, 'r') as zip_ref:
            zip_ref.extractall(".")
        print(f"Extracted: {f}")
    else:
        print(f"Not a zip (likely already extracted): {f.replace('.zip', '')}")


Extracted: articles.csv.zip
Extracted: customers.csv.zip
Extracted: transactions_train.csv.zip


### Load into pandas (same memory-efficient dtypes as the EDA notebook)

In [5]:
import pandas as pd

articles = pd.read_csv("articles.csv")
customers = pd.read_csv("customers.csv")
transactions = pd.read_csv(
    "transactions_train.csv",
    dtype={"article_id": "int32", "customer_id": "str", "price": "float32", "sales_channel_id": "int8"},
    parse_dates=["t_dat"]
)

print("Articles:", articles.shape)
print("Customers:", customers.shape)
print("Transactions:", transactions.shape)


Articles: (105542, 25)
Customers: (1371980, 7)
Transactions: (31788324, 5)


### Create output directory for all Step 3 artifacts

In [6]:
import os
from google.colab import drive

drive.mount('/content/drive')

# Since hm_recsys already exists in your Drive, save everything there directly
PROCESSED_DATA_DIR = "/content/drive/MyDrive/hm_recsys/processed_data"
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
print("Artifacts will be saved under:", PROCESSED_DATA_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Artifacts will be saved under: /content/drive/MyDrive/hm_recsys/processed_data


---
## Part A: Data Cleaning

**Why this step matters:** every downstream model (content-based, collaborative, hybrid) will
inherit any garbage left in the raw data. Cleaning is done once here so every later notebook
works from a trustworthy, consistent base — this is the entire point of a dedicated
preprocessing stage instead of re-cleaning ad hoc inside each model notebook.


**A.1 Track dataset sizes before cleaning (for the report)**

In [7]:
cleaning_report = {}

cleaning_report["before"] = {
    "articles_rows": len(articles),
    "customers_rows": len(customers),
    "transactions_rows": len(transactions)
}
print(cleaning_report["before"])


{'articles_rows': 105542, 'customers_rows': 1371980, 'transactions_rows': 31788324}


**A.2 Remove duplicate records**

*Why:* Exact duplicate rows add no new information and would silently inflate purchase counts
and popularity metrics, biasing both content-based and collaborative signals.


In [8]:
dupe_articles = articles.duplicated().sum()
dupe_customers = customers.duplicated().sum()
dupe_transactions = transactions.duplicated().sum()

articles = articles.drop_duplicates().reset_index(drop=True)
customers = customers.drop_duplicates().reset_index(drop=True)
transactions = transactions.drop_duplicates().reset_index(drop=True)

cleaning_report["duplicates_removed"] = {
    "articles": int(dupe_articles),
    "customers": int(dupe_customers),
    "transactions": int(dupe_transactions)
}
print(cleaning_report["duplicates_removed"])


{'articles': 0, 'customers': 0, 'transactions': 2974905}


**A.3 Handle missing values**

*Why each choice:*
- `detail_desc` (articles): missing description → fill with empty string, NOT drop the row.
  Dropping would lose a real, purchasable product just because its text metadata is incomplete —
  the product still needs to exist for collaborative filtering even if content-based has less to work with for it.
- `age` (customers): missing age → impute with the median age. Age isn't used for CF/CB core
  logic yet, but keeping the row (instead of dropping the customer) preserves their transaction history.
- `FN`, `Active`, `club_member_status`, `fashion_news_frequency` (customers): missing → fill with
  a explicit "UNKNOWN" / 0, since these are categorical/flag fields where "no data" is itself
  meaningful information (e.g., customer never opted into marketing, so no flag was ever set).


In [9]:
def handle_missing_articles(df):
    df = df.copy()
    missing_desc_count = df["detail_desc"].isnull().sum()
    df["detail_desc"] = df["detail_desc"].fillna("")
    return df, missing_desc_count

def handle_missing_customers(df):
    df = df.copy()
    missing_age_count = df["age"].isnull().sum()
    median_age = df["age"].median()
    df["age"] = df["age"].fillna(median_age)

    for col in ["FN", "Active"]:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    for col in ["club_member_status", "fashion_news_frequency"]:
        if col in df.columns:
            df[col] = df[col].fillna("UNKNOWN")
    return df, missing_age_count

articles, missing_desc_count = handle_missing_articles(articles)
customers, missing_age_count = handle_missing_customers(customers)

cleaning_report["missing_values_handled"] = {
    "articles_detail_desc_filled": int(missing_desc_count),
    "customers_age_imputed_with_median": int(missing_age_count)
}
print(cleaning_report["missing_values_handled"])

# Confirm no unexpected missing values remain in key columns
print("\nRemaining missing values check:")
print("Articles key cols:", articles[["article_id", "prod_name", "detail_desc"]].isnull().sum().to_dict())
print("Customers key cols:", customers[["customer_id", "age"]].isnull().sum().to_dict())
print("Transactions key cols:", transactions[["customer_id", "article_id", "t_dat", "price"]].isnull().sum().to_dict())


{'articles_detail_desc_filled': 416, 'customers_age_imputed_with_median': 15861}

Remaining missing values check:
Articles key cols: {'article_id': 0, 'prod_name': 0, 'detail_desc': 0}
Customers key cols: {'customer_id': 0, 'age': 0}
Transactions key cols: {'customer_id': 0, 'article_id': 0, 't_dat': 0, 'price': 0}


**A.4 Handle inconsistent datatypes**

*Why:* `customer_id` must stay a string (it's a long hex hash — converting it to a number would
lose precision/truncate it). `article_id` must be a consistent integer type across all three
tables, or merges between `transactions` and `articles` will silently fail to match rows.


In [10]:
def enforce_dtypes(articles, customers, transactions):
    articles = articles.copy()
    customers = customers.copy()
    transactions = transactions.copy()

    articles["article_id"] = articles["article_id"].astype("int32")
    transactions["article_id"] = transactions["article_id"].astype("int32")

    customers["customer_id"] = customers["customer_id"].astype("str")
    transactions["customer_id"] = transactions["customer_id"].astype("str")

    transactions["price"] = transactions["price"].astype("float32")

    return articles, customers, transactions

articles, customers, transactions = enforce_dtypes(articles, customers, transactions)
print("Dtype check:")
print("articles.article_id:", articles["article_id"].dtype)
print("transactions.article_id:", transactions["article_id"].dtype)
print("customers.customer_id:", customers["customer_id"].dtype)
print("transactions.customer_id:", transactions["customer_id"].dtype)


Dtype check:
articles.article_id: int32
transactions.article_id: int32
customers.customer_id: object
transactions.customer_id: object


**A.5 Convert dates to datetime format**

*Why:* `t_dat` was already parsed via `parse_dates` at load time, but we verify explicitly here
because any later CSV re-load (e.g. in a different notebook) would silently read it back as a
plain string unless we're careful — a very common source of bugs in recsys pipelines that rely
on chronological ordering (like our time-based train/test split in Part F).


In [11]:
transactions["t_dat"] = pd.to_datetime(transactions["t_dat"], errors="coerce")

invalid_dates = transactions["t_dat"].isnull().sum()
print(f"Rows with unparseable dates: {invalid_dates}")
print("Date range:", transactions["t_dat"].min(), "to", transactions["t_dat"].max())

cleaning_report["invalid_dates_found"] = int(invalid_dates)


Rows with unparseable dates: 0
Date range: 2018-09-20 00:00:00 to 2020-09-22 00:00:00


**A.6 Remove invalid transactions & check impossible values**

*Why:* A transaction with price <= 0 is either a data-entry error or a return/refund record
misfiled as a purchase — either way, it doesn't represent genuine purchase intent and would
distort both popularity counts and any price-based features. Similarly, ages outside a plausible
human range (e.g. negative, or absurdly high like 120+) indicate data entry errors, not real customers.


In [12]:
# Invalid transactions: non-positive price, or missing critical IDs
invalid_price_mask = transactions["price"] <= 0
missing_id_mask = transactions["customer_id"].isnull() | transactions["article_id"].isnull()

n_invalid_price = invalid_price_mask.sum()
n_missing_ids = missing_id_mask.sum()

transactions = transactions[~invalid_price_mask & ~missing_id_mask].reset_index(drop=True)

# Impossible age values in customers
impossible_age_mask = (customers["age"] < 10) | (customers["age"] > 100)
n_impossible_age = impossible_age_mask.sum()
customers.loc[impossible_age_mask, "age"] = customers["age"].median()

cleaning_report["invalid_transactions_removed"] = {
    "non_positive_price": int(n_invalid_price),
    "missing_ids": int(n_missing_ids)
}
cleaning_report["impossible_ages_corrected"] = int(n_impossible_age)

print(cleaning_report["invalid_transactions_removed"])
print("Impossible ages corrected:", cleaning_report["impossible_ages_corrected"])


{'non_positive_price': 0, 'missing_ids': 0}
Impossible ages corrected: 0


**A.7 Verify dataset integrity after cleaning**

In [13]:
integrity_checks = {
    "articles_duplicate_ids": int(articles["article_id"].duplicated().sum()),
    "customers_duplicate_ids": int(customers["customer_id"].duplicated().sum()),
    "transactions_null_customer_id": int(transactions["customer_id"].isnull().sum()),
    "transactions_null_article_id": int(transactions["article_id"].isnull().sum()),
    "transactions_negative_price": int((transactions["price"] < 0).sum()),
    "transactions_orphan_articles": int((~transactions["article_id"].isin(articles["article_id"])).sum()),
    "transactions_orphan_customers": int((~transactions["customer_id"].isin(customers["customer_id"])).sum()),
}
print("Integrity checks (all should read 0):")
for k, v in integrity_checks.items():
    print(f"  {k}: {v}")

cleaning_report["integrity_checks_after_cleaning"] = integrity_checks


Integrity checks (all should read 0):
  articles_duplicate_ids: 0
  customers_duplicate_ids: 0
  transactions_null_customer_id: 0
  transactions_null_article_id: 0
  transactions_negative_price: 0
  transactions_orphan_articles: 0
  transactions_orphan_customers: 0


**A.8 Cleaning report summary**

In [14]:
cleaning_report["after"] = {
    "articles_rows": len(articles),
    "customers_rows": len(customers),
    "transactions_rows": len(transactions)
}

print("="*60)
print("CLEANING REPORT")
print("="*60)
for section, content in cleaning_report.items():
    print(f"\n{section}:")
    print(content)


CLEANING REPORT

before:
{'articles_rows': 105542, 'customers_rows': 1371980, 'transactions_rows': 31788324}

duplicates_removed:
{'articles': 0, 'customers': 0, 'transactions': 2974905}

missing_values_handled:
{'articles_detail_desc_filled': 416, 'customers_age_imputed_with_median': 15861}

invalid_dates_found:
0

invalid_transactions_removed:
{'non_positive_price': 0, 'missing_ids': 0}

impossible_ages_corrected:
0

integrity_checks_after_cleaning:
{'articles_duplicate_ids': 0, 'customers_duplicate_ids': 0, 'transactions_null_customer_id': 0, 'transactions_null_article_id': 0, 'transactions_negative_price': 0, 'transactions_orphan_articles': 0, 'transactions_orphan_customers': 0}

after:
{'articles_rows': 105542, 'customers_rows': 1371980, 'transactions_rows': 28813419}


---
## Part B: Product Feature Engineering (Content-Based Prep)

**Why:** content-based filtering needs a single, clean text blob per product that captures
"what this product is" in natural language, so a TF-IDF vectorizer (in a later notebook) can
measure similarity between products based on shared words. We build that text field here —
**vectorization itself is deliberately deferred to the modeling notebook.**


**B.1 Build combined_features text column**

In [15]:
import re

def clean_text(text):
    """Lowercase, strip punctuation, collapse extra whitespace.
    This normalization ensures TF-IDF later treats 'Black' and 'black.' as the same token,
    rather than as two different tokens due to case/punctuation noise."""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # remove punctuation/special characters
    text = re.sub(r"\s+", " ", text).strip()   # collapse multiple spaces into one
    return text

def build_combined_features(df):
    """Concatenate the key descriptive fields into one text blob per product."""
    df = df.copy()

    text_columns = ["prod_name", "product_type_name", "product_group_name",
                     "department_name", "garment_group_name", "colour_group_name",
                     "detail_desc"]

    # Missing descriptions were already filled with "" in Part A, but we double-guard here
    for col in text_columns:
        df[col] = df[col].fillna("").astype(str)

    df["combined_features_raw"] = df[text_columns].agg(" ".join, axis=1)
    df["combined_features"] = df["combined_features_raw"].apply(clean_text)

    return df

articles = build_combined_features(articles)

print("Example combined_features:")
display(articles[["article_id", "prod_name", "combined_features"]].head(5))


Example combined_features:


,article_id,prod_name,combined_features
0,108775015,Strap top,strap top vest top garment upper body jersey b...
1,108775044,Strap top,strap top vest top garment upper body jersey b...
2,108775051,Strap top (1),strap top 1 vest top garment upper body jersey...
3,110065001,OP T-shirt (Idro),op t shirt idro bra underwear clean lingerie u...
4,110065002,OP T-shirt (Idro),op t shirt idro bra underwear clean lingerie u...


**B.2 Sanity check the text field**

*Why:* an empty or near-empty `combined_features` string for a product means content-based
similarity will have nothing to work with for it — worth flagging now rather than discovering
it silently during modeling.


In [16]:
empty_features = (articles["combined_features"].str.len() == 0).sum()
avg_length = articles["combined_features"].apply(lambda x: len(x.split())).mean()

print(f"Products with empty combined_features: {empty_features}")
print(f"Average word count per combined_features entry: {avg_length:.1f}")


Products with empty combined_features: 0
Average word count per combined_features entry: 36.9


---
## Part C: Customer Feature Engineering

**Why build customer-level features at all:** these aren't used for the core CF/CB similarity
math directly, but they're valuable for cold-start handling, business reporting, and as
auxiliary features a hybrid model can use to weight recommendations (e.g., a highly recent,
frequent buyer might get more collaborative-filtering weight; a rare buyer might lean more on
content-based).


In [17]:
def build_customer_features(transactions, articles):
    """Aggregate transaction history into one row of features per customer.
    Rewritten to avoid row-wise .apply() and per-group lambda aggregation,
    which do not scale to 1M+ customers — everything below is vectorized."""

    reference_date = transactions["t_dat"].max()

    grouped = transactions.groupby("customer_id").agg(
        total_purchases=("article_id", "count"),
        first_purchase_date=("t_dat", "min"),
        last_purchase_date=("t_dat", "max")
    ).reset_index()

    grouped["customer_lifetime_days"] = (
        grouped["last_purchase_date"] - grouped["first_purchase_date"]
    ).dt.days

    # Vectorized: np.maximum works element-wise across the whole column at once,
    # instead of calling a Python function per row
    lifetime_months = np.maximum(grouped["customer_lifetime_days"] / 30, 1)
    grouped["avg_purchases_per_month"] = grouped["total_purchases"] / lifetime_months

    grouped["recency_days"] = (reference_date - grouped["last_purchase_date"]).dt.days

    lifetime_days_safe = np.maximum(grouped["customer_lifetime_days"], 1)
    grouped["purchase_frequency"] = grouped["total_purchases"] / lifetime_days_safe

    # Vectorized "most frequent value per group" trick:
    # 1. count occurrences of each (customer_id, category) pair
    # 2. sort so the highest count per customer comes first
    # 3. drop_duplicates keeps only that top row per customer
    # This avoids calling value_counts() separately for each of 1.36M customers.
    txn_with_meta = transactions.merge(
        articles[["article_id", "product_group_name", "department_name"]],
        on="article_id", how="left"
    )

    cat_counts = (
        txn_with_meta.groupby(["customer_id", "product_group_name"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    fav_category = cat_counts.drop_duplicates(subset="customer_id", keep="first")[
        ["customer_id", "product_group_name"]
    ].rename(columns={"product_group_name": "favourite_category"})

    dept_counts = (
        txn_with_meta.groupby(["customer_id", "department_name"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    fav_department = dept_counts.drop_duplicates(subset="customer_id", keep="first")[
        ["customer_id", "department_name"]
    ].rename(columns={"department_name": "favourite_department"})

    customer_features = grouped.merge(fav_category, on="customer_id", how="left")
    customer_features = customer_features.merge(fav_department, on="customer_id", how="left")

    return customer_features

import numpy as np

customer_features = build_customer_features(transactions, articles)
print("Customer feature table shape:", customer_features.shape)
display(customer_features.head(5))

Customer feature table shape: (1362281, 10)


,customer_id,total_purchases,first_purchase_date,last_purchase_date,customer_lifetime_days,avg_purchases_per_month,recency_days,purchase_frequency,favourite_category,favourite_department
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,19,2018-12-27,2020-09-05,618,0.922330,17,0.030744,Garment Upper body,Suit
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,78,2018-09-21,2020-07-08,656,3.567073,76,0.118902,Garment Upper body,Swimwear
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,15,2018-09-20,2020-09-15,726,0.619835,7,0.020661,Garment Upper body,Swimwear
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,2,2019-06-09,2019-06-09,0,2.000000,471,2.000000,Underwear,Ladies Sport Bras
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,13,2018-10-12,2020-08-12,670,0.582090,41,0.019403,Garment Upper body,Swimwear


**Why each feature helps recommendations:**
- **`total_purchases`** — separates highly engaged users (good CF signal) from sparse/cold-start users
- **`first_purchase_date` / `last_purchase_date`** — needed to compute lifetime and recency, and for the time-based train/test split in Part F
- **`customer_lifetime_days`** — normalizes activity; a customer active 2 days with 5 purchases is very different from one active 200 days with 5 purchases
- **`avg_purchases_per_month` / `purchase_frequency`** — proxies for engagement intensity, useful for weighting a hybrid blend
- **`recency_days`** — classic RFM (Recency-Frequency-Monetary) signal; recently active customers are more likely to respond to recommendations
- **`favourite_category` / `favourite_department`** — a simple, interpretable content-based fallback signal (e.g., recommend more from a customer's favourite category when collaborative signal is weak)


---
## Part D: Interaction Feature Engineering

**Why implicit feedback differs from explicit ratings:** an explicit rating (like a 1-5 star
review) directly tells us *how much* a user liked something, including negative signal (a 1-star
rating is informative). A purchase only tells us a user *acted* — it says nothing about
satisfaction, and, critically, **the absence of a purchase is not a real negative signal** (a
customer might love a product they simply never saw). This means we can't reuse rating-based
metrics (like RMSE against a 1-5 scale) later — implicit-feedback models need are evaluated
differently (e.g., ranking metrics like Precision@K, Recall@K), which we'll address in the modeling stage, not here.


In [18]:
def build_interaction_dataset(transactions):
    """Aggregate raw transaction rows into one row per (customer, article) pair."""

    interactions = transactions.groupby(["customer_id", "article_id"]).agg(
        purchase_count=("t_dat", "count"),
        last_purchase_date=("t_dat", "max")
    ).reset_index()

    # Binary implicit feedback signal: 1 = interacted (purchased at least once)
    interactions["interaction"] = 1

    reference_date = transactions["t_dat"].max()
    interactions["days_since_last_purchase"] = (
        reference_date - interactions["last_purchase_date"]
    ).dt.days

    return interactions

interaction_dataset = build_interaction_dataset(transactions)
print("Interaction dataset shape:", interaction_dataset.shape)
display(interaction_dataset.head(5))


Interaction dataset shape: (27306439, 6)


,customer_id,article_id,purchase_count,last_purchase_date,interaction,days_since_last_purchase
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,176209023,1,2018-12-27,1,635
1,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601006,1,2019-05-25,1,486
2,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,568601043,1,2020-09-05,1,17
3,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,607642008,1,2019-07-25,1,425
4,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,625548001,1,2018-12-27,1,635


---
## Part E: Sparse User-Item Matrix

**Why sparse matrices are required, and why dense is infeasible:** with ~1.37M customers and
~105K articles, a dense matrix would need roughly 1.37M x 105K ≈ 144 billion cells. Even storing
each cell as a single byte, that's ~144GB — far beyond Colab's RAM (12-25GB depending on tier).
A sparse matrix instead stores only the non-zero (i.e., actually-occurred) entries, which for
this dataset is only tens of millions — a difference of several orders of magnitude in memory.


In [19]:
from scipy.sparse import csr_matrix, save_npz
from sklearn.preprocessing import LabelEncoder
import pickle

def build_sparse_matrix(interaction_dataset):
    """Encode string IDs to integer indices and build a CSR sparse matrix."""
    user_encoder = LabelEncoder()
    item_encoder = LabelEncoder()

    interaction_dataset = interaction_dataset.copy()
    interaction_dataset["user_idx"] = user_encoder.fit_transform(interaction_dataset["customer_id"])
    interaction_dataset["item_idx"] = item_encoder.fit_transform(interaction_dataset["article_id"])

    n_users = interaction_dataset["user_idx"].nunique()
    n_items = interaction_dataset["item_idx"].nunique()

    sparse_matrix = csr_matrix(
        (
            interaction_dataset["purchase_count"].values,   # use purchase_count as the cell value
            (interaction_dataset["user_idx"], interaction_dataset["item_idx"])
        ),
        shape=(n_users, n_items)
    )

    return sparse_matrix, user_encoder, item_encoder, interaction_dataset

sparse_matrix, user_encoder, item_encoder, interaction_dataset = build_sparse_matrix(interaction_dataset)

print("Sparse matrix shape:", sparse_matrix.shape)
print("Non-zero entries:", sparse_matrix.nnz)
print(f"Memory used: {sparse_matrix.data.nbytes / (1024**2):.2f} MB")

dense_equivalent_gb = (sparse_matrix.shape[0] * sparse_matrix.shape[1]) / (1024**3)
print(f"A dense equivalent would need approximately {dense_equivalent_gb:.1f} GB (infeasible)")


Sparse matrix shape: (1362281, 104547)
Non-zero entries: 27306439
Memory used: 208.33 MB
A dense equivalent would need approximately 132.6 GB (infeasible)


**E.1 Save the sparse matrix and encoders for reuse in later notebooks**

In [20]:
save_npz(os.path.join(PROCESSED_DATA_DIR, "interaction_matrix.npz"), sparse_matrix)

with open(os.path.join(PROCESSED_DATA_DIR, "label_encoders.pkl"), "wb") as f:
    pickle.dump({"user_encoder": user_encoder, "item_encoder": item_encoder}, f)

print("Saved:", os.path.join(PROCESSED_DATA_DIR, "interaction_matrix.npz"))
print("Saved:", os.path.join(PROCESSED_DATA_DIR, "label_encoders.pkl"))


Saved: /content/drive/MyDrive/hm_recsys/processed_data/interaction_matrix.npz
Saved: /content/drive/MyDrive/hm_recsys/processed_data/label_encoders.pkl


---
## Part F: Train/Test Split (Time-Based)

**Why random train-test splitting is NOT suitable here:** a random split would scatter a single
customer's purchases randomly between train and test, meaning the model could end up "training"
on a purchase made *after* the one it's being tested on — this is data leakage from the future,
and it doesn't reflect how the system will actually be used in production (predicting *future*
purchases from *past* behavior). Recommendation systems must be evaluated the way they'll be
deployed: given everything up to a point in time, predict what comes next. That requires a
**time-based split** (or leave-one-out by holding out each customer's single most recent purchase).

We use a **time-based split**: transactions before a cutoff date go to train, transactions on/after
it go to test. This preserves chronological realism across the entire dataset.


In [21]:
def time_based_split(transactions, test_fraction_days=7):
    """Split by date: the most recent `test_fraction_days` of transactions become the test set."""
    max_date = transactions["t_dat"].max()
    cutoff_date = max_date - pd.Timedelta(days=test_fraction_days)

    train = transactions[transactions["t_dat"] <= cutoff_date].copy()
    test = transactions[transactions["t_dat"] > cutoff_date].copy()

    return train, test, cutoff_date

train_transactions, test_transactions, cutoff_date = time_based_split(transactions, test_fraction_days=7)

print(f"Cutoff date: {cutoff_date.date()}")
print(f"Train transactions: {len(train_transactions):,} ({train_transactions['t_dat'].min().date()} to {train_transactions['t_dat'].max().date()})")
print(f"Test transactions: {len(test_transactions):,} ({test_transactions['t_dat'].min().date()} to {test_transactions['t_dat'].max().date()})")


Cutoff date: 2020-09-15
Train transactions: 28,595,503 (2018-09-20 to 2020-09-15)
Test transactions: 217,916 (2020-09-16 to 2020-09-22)


**F.1 Aggregate train/test transactions into interaction-level rows (same shape as Part D)**

*Why re-aggregate instead of reusing the full `interaction_dataset`:* the full interaction
dataset in Part D was built across the *entire* time range — using it directly for train/test
would leak future purchase counts into "train" rows. Train and test interaction tables must each
be built only from their own date range.


In [22]:
train_interactions = build_interaction_dataset(train_transactions)
test_interactions = build_interaction_dataset(test_transactions)

print("Train interactions shape:", train_interactions.shape)
print("Test interactions shape:", test_interactions.shape)

# A realistic evaluation also filters test to only users/items seen in train,
# since we can't meaningfully evaluate collaborative filtering for a customer
# or product the model has never seen before (that's the cold-start case,
# handled separately by content-based filtering, not by this train/test split).
known_users = set(train_interactions["customer_id"])
known_items = set(train_interactions["article_id"])

test_interactions_filtered = test_interactions[
    test_interactions["customer_id"].isin(known_users) &
    test_interactions["article_id"].isin(known_items)
].reset_index(drop=True)

print(f"Test interactions after filtering to known users/items: {len(test_interactions_filtered):,} "
      f"(dropped {len(test_interactions) - len(test_interactions_filtered):,} cold-start rows)")


Train interactions shape: (27101148, 6)
Test interactions shape: (213728, 6)
Test interactions after filtering to known users/items: 189,845 (dropped 23,883 cold-start rows)


**F.2 Save train/test interaction files**

In [23]:
train_interactions.to_csv(os.path.join(PROCESSED_DATA_DIR, "train_interactions.csv"), index=False)
test_interactions_filtered.to_csv(os.path.join(PROCESSED_DATA_DIR, "test_interactions.csv"), index=False)

print("Saved:", os.path.join(PROCESSED_DATA_DIR, "train_interactions.csv"))
print("Saved:", os.path.join(PROCESSED_DATA_DIR, "test_interactions.csv"))


Saved: /content/drive/MyDrive/hm_recsys/processed_data/train_interactions.csv
Saved: /content/drive/MyDrive/hm_recsys/processed_data/test_interactions.csv


---
## Part G: Save Processed Data

Saving every reusable artifact now means later notebooks (content-based, collaborative, hybrid)
can simply load these files directly, instead of repeating cleaning/feature logic — keeping the
whole pipeline modular and consistent.


In [24]:
# Processed core tables
articles.to_csv(os.path.join(PROCESSED_DATA_DIR, "processed_articles.csv"), index=False)
customers.to_csv(os.path.join(PROCESSED_DATA_DIR, "processed_customers.csv"), index=False)
transactions.to_csv(os.path.join(PROCESSED_DATA_DIR, "processed_transactions.csv"), index=False)

# Content-based feature table (just the ID + combined_features, kept lean)
content_features = articles[["article_id", "combined_features"]].copy()
content_features.to_csv(os.path.join(PROCESSED_DATA_DIR, "content_features.csv"), index=False)

# Full aggregated interaction dataset (whole date range, for content-based/EDA reuse,
# NOT for train/test model evaluation — that uses the Part F files instead)
interaction_dataset.to_csv(os.path.join(PROCESSED_DATA_DIR, "interaction_dataset.csv"), index=False)

# Customer feature table
customer_features.to_csv(os.path.join(PROCESSED_DATA_DIR, "customer_features.csv"), index=False)

print("All processed files saved under:", PROCESSED_DATA_DIR)
!ls -lh "{PROCESSED_DATA_DIR}"


All processed files saved under: /content/drive/MyDrive/hm_recsys/processed_data
total 8.5G
-rw------- 1 root root  23M Aug  8 17:03 content_features.csv
-rw------- 1 root root 197M Aug  8 17:06 customer_features.csv
-rw------- 1 root root 2.8G Aug  8 17:06 interaction_dataset.csv
-rw------- 1 root root  70M Aug  8 16:58 interaction_matrix.npz
-rw------- 1 root root  88M Aug  8 16:58 label_encoders.pkl
-rw------- 1 root root  79M Aug  8 17:01 processed_articles.csv
-rw------- 1 root root 206M Aug  8 17:01 processed_customers.csv
-rw------- 1 root root 2.7G Aug  8 17:03 processed_transactions.csv
-rw------- 1 root root  17M Aug  8 17:01 test_interactions.csv
-rw------- 1 root root 2.4G Aug  8 17:01 train_interactions.csv


**What each saved file is for:**

| File | Purpose |
|---|---|
| `processed_articles.csv` | Cleaned product metadata — source for content-based filtering and any future feature updates |
| `processed_customers.csv` | Cleaned customer metadata — source for cold-start handling and demographic features |
| `processed_transactions.csv` | Cleaned raw transaction log — source of truth if any future feature needs to be recomputed |
| `content_features.csv` | Lean `article_id` + `combined_features` table — direct input to TF-IDF vectorization in the content-based notebook |
| `customer_features.csv` | Aggregated customer-level features (recency, frequency, favourites) — input for hybrid weighting logic |
| `interaction_dataset.csv` | Full aggregated (customer, article) interaction table across the whole date range — reference dataset, not for train/test evaluation |
| `train_interactions.csv` / `test_interactions.csv` | Time-based split — used to train and evaluate the collaborative filtering model |
| `interaction_matrix.npz` | Sparse user-item matrix — direct input to matrix factorization (ALS/SVD) in the collaborative filtering notebook |
| `label_encoders.pkl` | Saved `LabelEncoder` objects mapping `customer_id`/`article_id` to matrix indices — required to translate model outputs (matrix indices) back into real IDs |


---
## Part H: Validation

Final checks before declaring preprocessing complete — this is what prevents silent bugs in
later notebooks (e.g., an off-by-one in ID encoding that wouldn't surface until a model produces
nonsensical recommendations).


In [25]:
validation_report = {}

# 1. No duplicate interactions
validation_report["duplicate_interactions_full"] = int(
    interaction_dataset.duplicated(subset=["customer_id", "article_id"]).sum()
)
validation_report["duplicate_interactions_train"] = int(
    train_interactions.duplicated(subset=["customer_id", "article_id"]).sum()
)
validation_report["duplicate_interactions_test"] = int(
    test_interactions_filtered.duplicated(subset=["customer_id", "article_id"]).sum()
)

# 2. No missing IDs
validation_report["missing_customer_ids"] = int(interaction_dataset["customer_id"].isnull().sum())
validation_report["missing_article_ids"] = int(interaction_dataset["article_id"].isnull().sum())

# 3. Correct matrix dimensions (matches number of unique encoded users/items)
validation_report["matrix_shape"] = sparse_matrix.shape
validation_report["matrix_matches_encoders"] = (
    sparse_matrix.shape[0] == len(user_encoder.classes_) and
    sparse_matrix.shape[1] == len(item_encoder.classes_)
)

# 4. Consistent user/product mappings: spot check a round-trip encode/decode
sample_user = interaction_dataset["customer_id"].iloc[0]
encoded = user_encoder.transform([sample_user])[0]
decoded = user_encoder.inverse_transform([encoded])[0]
validation_report["encoder_roundtrip_ok"] = (sample_user == decoded)

print("Validation report:")
for k, v in validation_report.items():
    print(f"  {k}: {v}")


Validation report:
  duplicate_interactions_full: 0
  duplicate_interactions_train: 0
  duplicate_interactions_test: 0
  missing_customer_ids: 0
  missing_article_ids: 0
  matrix_shape: (1362281, 104547)
  matrix_matches_encoders: True
  encoder_roundtrip_ok: True


**H.1 Verify all saved files can be reloaded successfully**

In [ ]:
import gc
import pandas as pd

def check_file(path, loader=pd.read_csv, **kwargs):
    """Load a file just to confirm it works, print its shape, then immediately
    free the memory — never keep more than one reloaded file in RAM at a time."""
    try:
        df = loader(path, **kwargs)
        shape = df.shape
        del df
        gc.collect()
        print(f"{path}: OK, shape={shape}")
    except Exception as e:
        print(f"{path}: FAILED — {e}")

check_file(os.path.join(PROCESSED_DATA_DIR, "processed_articles.csv"))
check_file(os.path.join(PROCESSED_DATA_DIR, "processed_customers.csv"))
check_file(os.path.join(PROCESSED_DATA_DIR, "processed_transactions.csv"))
check_file(os.path.join(PROCESSED_DATA_DIR, "content_features.csv"))
check_file(os.path.join(PROCESSED_DATA_DIR, "interaction_dataset.csv"))
check_file(os.path.join(PROCESSED_DATA_DIR, "customer_features.csv"))
check_file(os.path.join(PROCESSED_DATA_DIR, "train_interactions.csv"))
check_file(os.path.join(PROCESSED_DATA_DIR, "test_interactions.csv"))

from scipy.sparse import load_npz
mat = load_npz(os.path.join(PROCESSED_DATA_DIR, "interaction_matrix.npz"))
print(f"interaction_matrix.npz: OK, shape={mat.shape}, nnz={mat.nnz}")
del mat
gc.collect()

import pickle
with open(os.path.join(PROCESSED_DATA_DIR, "label_encoders.pkl"), "rb") as f:
    enc = pickle.load(f)
print(f"label_encoders.pkl: OK, keys={list(enc.keys())}")
del enc
gc.collect()


/content/drive/MyDrive/hm_recsys/processed_data/processed_articles.csv: OK, shape=(105542, 27)
/content/drive/MyDrive/hm_recsys/processed_data/processed_customers.csv: OK, shape=(1371980, 7)


In [ ]:
import pandas as pd
import pickle

with open(os.path.join(PROCESSED_DATA_DIR, "label_encoders.pkl"), "rb") as f:
    encoders = pickle.load(f)

user_encoder = encoders["user_encoder"]
item_encoder = encoders["item_encoder"]

def encode_and_save_lightweight(input_csv, output_parquet):
    """Reload a heavy string-ID file, replace IDs with compact integers,
    and save as Parquet — dramatically smaller and faster to reload later."""
    df = pd.read_csv(input_csv)
    df["user_idx"] = user_encoder.transform(df["customer_id"])
    df["item_idx"] = item_encoder.transform(df["article_id"])
    df = df.drop(columns=["customer_id", "article_id"])
    df.to_parquet(output_parquet, index=False)
    print(f"{output_parquet}: {df.shape}")

encode_and_save_lightweight(
    os.path.join(PROCESSED_DATA_DIR, "train_interactions.csv"),
    os.path.join(PROCESSED_DATA_DIR, "train_interactions_encoded.parquet")
)
encode_and_save_lightweight(
    os.path.join(PROCESSED_DATA_DIR, "test_interactions.csv"),
    os.path.join(PROCESSED_DATA_DIR, "test_interactions_encoded.parquet")
)


In [ ]:
import gc
import pandas as pd

def check_file(path, loader=pd.read_parquet):
    try:
        df = loader(path)
        shape = df.shape
        del df
        gc.collect()
        print(f"{path}: OK, shape={shape}")
    except Exception as e:
        print(f"{path}: FAILED — {e}")

check_file(os.path.join(PROCESSED_DATA_DIR, "train_interactions_encoded.parquet"))
check_file(os.path.join(PROCESSED_DATA_DIR, "test_interactions_encoded.parquet"))


---
## Deliverable Summary

1. **Preprocessing performed:** duplicate removal, missing-value handling, dtype enforcement,
   invalid-transaction removal, impossible-value correction, datetime conversion, and full
   integrity verification — all logged in the cleaning report.

2. **New features created:**
   - Product: `combined_features` (cleaned, concatenated text for content-based similarity)
   - Customer: recency/frequency/lifetime/favourite-category features (RFM-style, for cold-start and hybrid weighting)
   - Interaction: aggregated `purchase_count`, `days_since_last_purchase`, binary `interaction` flag

3. **Files saved:** all 9 artifacts listed in Part G/H, under `processed_data/`.

4. **How these will be used next:**
   - **Content-Based Filtering:** `content_features.csv` → TF-IDF vectorization → item-item similarity
   - **Collaborative Filtering:** `interaction_matrix.npz` + `label_encoders.pkl` + `train_interactions.csv`/`test_interactions.csv` → matrix factorization (ALS/SVD), evaluated with ranking metrics
   - **Hybrid Recommendation:** `customer_features.csv` (activity level) used to decide, per user, how much to weight content-based vs. collaborative scores — e.g., low-activity users lean more on content-based, high-activity users lean more on collaborative

**Stopping point confirmed:** no recommendation algorithms have been built in this notebook —
preprocessing and feature engineering only.
